# Retrieval Benchmark Analysis

Answers to the 7 analysis questions using benchmark results from `results/`.

**Run `scripts/run_benchmark.py` and `scripts/run_hyde_subset.py` before this notebook.**

In [ ]:
import sys
from pathlib import Path

# Allow imports from project root
PROJECT_DIR = Path("../").resolve()
sys.path.insert(0, str(PROJECT_DIR))

import json
import pandas as pd
import numpy as np

from config import RESULTS_DIR, CACHE_DIR, SEED
from src.analysis.disagreement import (
    find_bm25_beats_dense,
    find_max_divergence_queries,
    gold_rank,
)
from src.analysis.qualitative import format_query_comparison, dump_disagreement_examples

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
# Load main results tables
msmarco_df = pd.read_csv(RESULTS_DIR / "msmarco_results.csv", index_col="method")
scifact_df = pd.read_csv(RESULTS_DIR / "scifact_results.csv", index_col="method")

print("MS MARCO:")
display(msmarco_df)
print("\nSciFact:")
display(scifact_df)

In [ ]:
# Helper: load per-query CSV as a run dict {qid: {doc_id: score}}
def load_run(method: str, dataset: str) -> dict:
    path = RESULTS_DIR / "per_query" / f"{method}_{dataset}.csv"
    if not path.exists():
        return {}
    df = pd.read_csv(path)
    run = {}
    for _, row in df.iterrows():
        qid = str(row["query_id"])
        run.setdefault(qid, {})[str(row["doc_id"])] = float(row["score"])
    return run

# Helper: build qrels dict from per-query CSV
def load_qrels(method: str, dataset: str) -> dict:
    path = RESULTS_DIR / "per_query" / f"{method}_{dataset}.csv"
    if not path.exists():
        return {}
    df = pd.read_csv(path)
    qrels = {}
    for _, row in df[df["is_relevant"] == 1].iterrows():
        qid = str(row["query_id"])
        qrels.setdefault(qid, {})[str(row["doc_id"])] = 1
    return qrels

# Helper: load query texts
def load_query_texts(method: str, dataset: str) -> dict:
    path = RESULTS_DIR / "per_query" / f"{method}_{dataset}.csv"
    if not path.exists():
        return {}
    df = pd.read_csv(path).drop_duplicates("query_id")
    return dict(zip(df["query_id"].astype(str), df["query_text"]))

---
## Q1: Overall Winner

Which method wins per metric? Do rankings flip between datasets?

In [ ]:
metric_cols = [c for c in msmarco_df.columns if c != "Latency_ms"]

print("=== MS MARCO: Best method per metric ===")
for col in metric_cols:
    best = msmarco_df[col].idxmax()
    print(f"  {col:12s}: {best} ({msmarco_df.loc[best, col]:.4f})")

print("\n=== SciFact: Best method per metric ===")
for col in metric_cols:
    best = scifact_df[col].idxmax()
    print(f"  {col:12s}: {best} ({scifact_df.loc[best, col]:.4f})")

In [ ]:
# Rank each method per metric (rank 1 = best) on NDCG@10
ndcg_msmarco = msmarco_df["NDCG@10"].rank(ascending=False).rename("Rank_MSMARCO")
ndcg_scifact = scifact_df["NDCG@10"].rank(ascending=False).rename("Rank_SciFact")
ranking_comparison = pd.concat([ndcg_msmarco, ndcg_scifact], axis=1)
ranking_comparison["Rank_flip"] = (ranking_comparison["Rank_MSMARCO"] - ranking_comparison["Rank_SciFact"]).abs() >= 2
print("NDCG@10 ranking comparison (rank 1 = best):")
display(ranking_comparison.sort_values("Rank_MSMARCO"))

**Interpretation:** Summarize which method wins most metrics overall, and any cases where a method performs well on one dataset but poorly on the other (rank flip = `True`).

---
## Q2: BM25 vs Dense (MiniLM)

Find 3–5 queries where BM25 beats Dense-MiniLM. Hypothesize why.

In [ ]:
for dataset in ["msmarco", "scifact"]:
    bm25_run = load_run("BM25", dataset)
    dense_run = load_run("Dense-MiniLM", dataset)
    qrels = load_qrels("BM25", dataset)
    query_texts = load_query_texts("BM25", dataset)

    if not bm25_run:
        print(f"No per-query data for {dataset} — run the benchmark first.")
        continue

    wins = find_bm25_beats_dense(bm25_run, dense_run, qrels, n=5)

    print(f"\n=== {dataset.upper()}: Queries where BM25 beats Dense-MiniLM ===")
    for _, row in wins.iterrows():
        qid = row["query_id"]
        text = query_texts.get(str(qid), "<not found>")
        print(f"  Query {qid}: '{text}'")
        print(f"    BM25 gold-rank={row['bm25_rank']}, Dense gold-rank={row['dense_rank']}, advantage={row['advantage']}")

**Hypotheses for BM25 wins:**
- Rare technical terms (exact lexical match required)
- Short queries with high-IDF keywords that dense models dilute
- Named entities or codes (e.g., drug names, gene IDs) not well-represented in MiniLM embeddings

---
## Q3: Top-vs-Worst Disagreement

Find 5 queries with maximum spread in gold-rank across all 7 methods.

In [ ]:
METHOD_NAMES = ["BM25", "TF-IDF", "Dense-MiniLM", "Dense-msmarco", "Hybrid-RRF", "CrossEncoder", "ColBERT"]

for dataset in ["msmarco", "scifact"]:
    runs = {}
    for m in METHOD_NAMES:
        r = load_run(m, dataset)
        if r:
            runs[m] = r

    qrels = load_qrels("BM25", dataset)
    query_texts = load_query_texts("BM25", dataset)

    if not runs:
        print(f"No per-query data for {dataset}.")
        continue

    divergence = find_max_divergence_queries(runs, qrels, n=5)

    print(f"\n=== {dataset.upper()}: Highest gold-rank spread across methods ===")
    for _, row in divergence.iterrows():
        qid = str(row["query_id"])
        text = query_texts.get(qid, "<not found>")
        print(f"  Query {qid} (spread={row['spread']}): '{text}'")
        print(f"    best_method={row['best_method']} (rank {row['best_rank']}), "
              f"worst_method={row['worst_method']} (rank {row['worst_rank']})")

**Interpretation:** High spread indicates queries where retrieval paradigm matters most. Inspect query text for clues: vocabulary mismatch, need for semantic paraphrase, or domain-specific syntax.

---
## Q4: HyDE Win/Fail Analysis

Inspect hypothetical quality for queries HyDE helped vs hurt.

In [ ]:
for dataset in ["msmarco", "scifact"]:
    hyp_path = CACHE_DIR / "hyde" / f"{dataset}__hypotheticals.json"
    hyde_run = load_run("HyDE", dataset)
    dense_run = load_run("Dense-MiniLM", dataset)
    qrels = load_qrels("BM25", dataset)

    if not hyp_path.exists():
        print(f"No hypotheticals found for {dataset} — run run_hyde_subset.py first.")
        continue

    hypotheticals = json.loads(hyp_path.read_text(encoding="utf-8"))
    print(f"\n=== {dataset.upper()}: {len(hypotheticals)} hypotheticals ===")

    # Compare HyDE vs Dense-MiniLM gold-rank per query
    comparison = []
    for qid, hyp_data in hypotheticals.items():
        if qid not in hyde_run or qid not in dense_run:
            continue
        hyde_r = gold_rank(qid, qrels, hyde_run)
        dense_r = gold_rank(qid, qrels, dense_run)
        comparison.append({
            "query_id": qid,
            "query": hyp_data.get("query", ""),
            "hypothetical": hyp_data.get("hypothetical", ""),
            "hyde_rank": hyde_r,
            "dense_rank": dense_r,
            "improvement": dense_r - hyde_r,  # positive = HyDE better
        })

    if not comparison:
        print("  No overlap between HyDE queries and per-query CSV — check run_hyde_subset.py output.")
        continue

    cdf = pd.DataFrame(comparison).sort_values("improvement", ascending=False)

    print("\n-- HyDE WINS (top 5) --")
    for _, row in cdf.head(5).iterrows():
        print(f"  Q: {row['query']}")
        print(f"  Hyp: {row['hypothetical'][:200]}..." if len(row['hypothetical']) > 200 else f"  Hyp: {row['hypothetical']}")
        print(f"  Dense rank={row['dense_rank']} → HyDE rank={row['hyde_rank']} (Δ={row['improvement']})")
        print()

    print("-- HyDE HURTS (bottom 5) --")
    for _, row in cdf.tail(5).iterrows():
        print(f"  Q: {row['query']}")
        print(f"  Hyp: {row['hypothetical'][:200]}..." if len(row['hypothetical']) > 200 else f"  Hyp: {row['hypothetical']}")
        print(f"  Dense rank={row['dense_rank']} → HyDE rank={row['hyde_rank']} (Δ={row['improvement']})")
        print()

**Hypotheses:**
- HyDE wins when the query is ambiguous and the hypothetical clarifies intent
- HyDE hurts when the model hallucinates irrelevant content or wrong domain terms
- Short factoid queries tend to benefit less (the hypothetical adds noise)

---
## Q5: M4 vs M3 — Domain Adaptation Benefit

Does `msmarco-distilbert` (M4) outperform `MiniLM` (M3) on MS MARCO but not SciFact?

In [ ]:
metrics_of_interest = ["P@1", "MRR@10", "NDCG@10", "MAP@100"]

print("=== M3 (Dense-MiniLM) vs M4 (Dense-msmarco) ===")
print("\nMS MARCO:")
comparison_msmarco = msmarco_df.loc[["Dense-MiniLM", "Dense-msmarco"], metrics_of_interest]
diff_msmarco = (comparison_msmarco.loc["Dense-msmarco"] - comparison_msmarco.loc["Dense-MiniLM"]).rename("M4 - M3")
display(pd.concat([comparison_msmarco, diff_msmarco.to_frame().T]))

print("\nSciFact:")
comparison_scifact = scifact_df.loc[["Dense-MiniLM", "Dense-msmarco"], metrics_of_interest]
diff_scifact = (comparison_scifact.loc["Dense-msmarco"] - comparison_scifact.loc["Dense-MiniLM"]).rename("M4 - M3")
display(pd.concat([comparison_scifact, diff_scifact.to_frame().T]))

print("\nConclusion:")
m4_wins_msmarco = (diff_msmarco > 0).sum()
m4_wins_scifact = (diff_scifact > 0).sum()
print(f"  M4 beats M3 on {m4_wins_msmarco}/{len(metrics_of_interest)} metrics for MS MARCO")
print(f"  M4 beats M3 on {m4_wins_scifact}/{len(metrics_of_interest)} metrics for SciFact")

**Expected finding:** M4 (`msmarco-distilbert`) was fine-tuned on MS MARCO triplets, so it should outperform M3 on that dataset. On SciFact (scientific text), the domain gap may cause M4 to lose its advantage or underperform M3.

---
## Q6: Hybrid M5 — Does RRF Beat Both Components?

Does Hybrid-RRF (M5) outperform both BM25 (M1) and Dense-MiniLM (M3) on every metric?

In [ ]:
metric_cols = [c for c in msmarco_df.columns if c != "Latency_ms"]

for dataset, df in [("MS MARCO", msmarco_df), ("SciFact", scifact_df)]:
    print(f"\n=== {dataset}: RRF vs components ===")
    methods = ["BM25", "Dense-MiniLM", "Hybrid-RRF"]
    subset = df.loc[methods, metric_cols]
    display(subset)

    rrf = subset.loc["Hybrid-RRF"]
    bm25 = subset.loc["BM25"]
    dense = subset.loc["Dense-MiniLM"]

    beats_bm25 = (rrf > bm25).sum()
    beats_dense = (rrf > dense).sum()
    total = len(metric_cols)

    print(f"  Hybrid-RRF beats BM25 on {beats_bm25}/{total} metrics")
    print(f"  Hybrid-RRF beats Dense-MiniLM on {beats_dense}/{total} metrics")
    if beats_bm25 == total and beats_dense == total:
        print("  → RRF strictly dominates both components")
    else:
        print("  → RRF does NOT strictly dominate on all metrics")
        failing = metric_cols[(rrf <= bm25) | (rrf <= dense)]
        print(f"  Metrics where RRF fails to beat at least one component: {list(failing)}")

**Expected finding:** RRF typically helps on recall-oriented metrics (R@k) and overall NDCG. It may not strictly dominate on P@1 if one component is already strong there. RRF trades precision for robustness.

---
## Q7: Re-Rank Cost-Benefit

Compare M3 Stage-1 R@100 vs CrossEncoder NDCG@10 gain vs latency overhead.

In [ ]:
for dataset, df in [("MS MARCO", msmarco_df), ("SciFact", scifact_df)]:
    print(f"\n=== {dataset}: Re-ranking cost-benefit ===")

    m3_row = df.loc["Dense-MiniLM"]
    ce_row = df.loc["CrossEncoder"]

    ndcg_gain = ce_row["NDCG@10"] - m3_row["NDCG@10"]
    mrr_gain = ce_row["MRR@10"] - m3_row["MRR@10"]
    latency_overhead = ce_row["Latency_ms"] - m3_row["Latency_ms"]
    latency_ratio = ce_row["Latency_ms"] / m3_row["Latency_ms"]

    print(f"  Stage-1 (M3) R@100: {m3_row.get('R@100', 'N/A')}")
    print(f"  NDCG@10:  M3={m3_row['NDCG@10']:.4f}  CE={ce_row['NDCG@10']:.4f}  Δ={ndcg_gain:+.4f}")
    print(f"  MRR@10:   M3={m3_row['MRR@10']:.4f}  CE={ce_row['MRR@10']:.4f}  Δ={mrr_gain:+.4f}")
    print(f"  Latency:  M3={m3_row['Latency_ms']:.1f} ms  CE={ce_row['Latency_ms']:.1f} ms  "
          f"overhead={latency_overhead:+.1f} ms ({latency_ratio:.1f}×)")

    if ndcg_gain > 0:
        print(f"  → Re-ranking improves NDCG@10 by {ndcg_gain:.4f} at {latency_ratio:.1f}× latency cost")
    else:
        print(f"  → Re-ranking does NOT improve NDCG@10 ({ndcg_gain:+.4f})")

**Interpretation:** The cross-encoder improves precision-oriented metrics (NDCG@10, MRR@10) when stage-1 R@100 is high (i.e., relevant docs are in the candidate set). High latency overhead is expected — the cross-encoder scores 100 query-doc pairs per query. In production, the first-stage k controls the tradeoff.